# Thommidhi tool-use evaluation
AI-assisted review draft; no submission or public publication. Run this notebook from the extracted bundle. It performs no network calls.
The original pilots preserve observed failures. The later checkpoint includes assisted recovery and batched cases; its results are not independent first-attempt success rates. Review semantic annotations in `build_checkpoint.py`.


In [ ]:
from pathlib import Path
import json, sys
root = Path.cwd()
if root.name == 'eval': root = root.parent
assert (root / 'eval/cases.json').exists(), 'Run from the extracted bundle root or eval folder'
sys.path.insert(0, str(root / 'eval'))
from score import evaluate
cases = json.loads((root / 'eval/cases.json').read_text())
for name in ['claude-pilot', 'codex-pilot', 'codex-checkpoint']:
    transcript = json.loads((root / f'evidence/{name}.json').read_text())
    report = evaluate(cases, transcript)
    print(name, report['summary'])


## Inspect failures and limitations
Scoring is deterministic, but semantic judgments are Codex-assisted annotations awaiting applicant review. Missing evidence stays unknown. A completed transport is not necessarily a committed write. Costs are unknown; shared batch timings cannot be summed as independent case latencies.


In [ ]:
for name in ['claude-pilot', 'codex-pilot']:
    transcript = json.loads((root / f'evidence/{name}.json').read_text())
    for row in evaluate(cases, transcript)['cases']:
        if row['status'] == 'fail':
            print(name, row['case_id'], row['source'])
            for check in row['checks']:
                if check['result'] == 'fail': print(check)


In [ ]:
import unittest
suite = unittest.defaultTestLoader.discover(str(root / 'eval'), pattern='test_*.py')
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()
